# contrastive_finetune: contrastive fine-tuning on top of the distilled main-line student

Start: the 1k-base student distilled from BGE-M3 + instructed 8K-embed (10,800 steps, lr 1e-4; TR-MTEB 67.71). Training: in-batch-negative InfoNCE on the (question, passage) pairs of the three QA datasets (`configs/contrastive_after_distill.yaml`), one epoch (~300 steps, a few minutes on an A100), with the **anchor** (distillation regulariser, cosine 10 / Gram 200 toward a frozen copy of the starting model) and lr 1e-5, one run per entry of `RUNS`.

Earlier rounds (TR-MTEB): plain InfoNCE 67.11 (forgetting), anchor 67.74, anchor + mined hard negatives 67.67; soups returned to ~67.8. None beat the distilled model by more than the 0.3 threshold.

**This round (advisor's idea, 11 Nov 2026): the answers.** The QA datasets also carry an answer per question (TurHistQuAD: a span, median 2 words; WikiRAG-TR: a generated sentence, median 32 words; Wikipedia-QA: a generated sentence). A second term pulls each question toward its own answer among the batch's answers (InfoNCE, identical answers masked) and is added with a per-pair weight `w`: `loss = InfoNCE(q, passage) + mean_i(w_i * InfoNCE_i(q, answer)) + anchor`. Runs: fixed `w` = 0.1 / 0.3 / 0.5, then dynamic: `w` grows from 0.1 to 0.5 with the answer's word count (full at 20 words) or character count (full at 150 characters), so a one-word answer ("1923") counts little. Same pairs, order and lr as the anchor run (`ctr_anchor_lr1e5`, 67.74): the answer term is the only change.

The answers cell downloads the three raw datasets (pinned revisions, ~650 MB, local disk) and attaches the answers to the existing filtered pairs (`attach_answers`; the pairs themselves stay as they are). Answers that equal a TR-MTEB / Mizan test text or share a 13-gram with one are removed (`benchmark_texts_all.jsonl`); the RAGTurk corpus is not in that list (its pairs were already filtered; an answer is a sentence from or about the pair's own passage). Read `pairs_train_clean_ans.stats.json`: matches per source should be ~100%.

Nothing here is a result: untuned starting values. Evaluate with `eval_trio` (TR-MTEB), Mizan and the RAG notebooks; compare against 67.74 (anchor alone, same everything else) and 67.71 (the distilled model); read only differences above ~0.3 (seed noise ~0.1, single seed). Soups are off this round (`SHARES = []`; the earlier soups moved scores by <0.1). Re-running after a disconnect resumes the unfinished run and skips finished ones. Delete this notebook when the results are in.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)



In [ ]:
# Answers: download the raw QA datasets (pinned revisions) to the local disk and attach their answers to the filtered
# pairs. CPU, a few minutes (most of it the 600 MB Wikipedia-QA parquet). Skipped when the output exists.
import os

PAIRS_CLEAN = f'{DRIVE_ROOT}/data/stage1/pairs_train_clean.jsonl'
PAIRS_ANS = f'{DRIVE_ROOT}/data/stage1/pairs_train_clean_ans.jsonl'
FORBIDDEN = f'{DRIVE_ROOT}/data/benchmark_texts_all.jsonl'   # TR-MTEB + Mizan test texts
assert os.path.exists(FORBIDDEN), FORBIDDEN
if os.path.exists(PAIRS_ANS):
    print('exists, skipped:', PAIRS_ANS)
else:
    get_ipython().system(f'python -m turkish_distill.attach_answers --pairs {PAIRS_CLEAN} --out {PAIRS_ANS} --raw-dir /content/raw --download --forbidden {FORBIDDEN}')
print(open(PAIRS_ANS.replace('.jsonl', '.stats.json')).read())

In [ ]:
# Contrastive runs: (name, extra overrides). A later override wins. All runs: anchor 10 / 200, lr 1e-5, the answered pairs.
DISTILLED = f'{DRIVE_ROOT}/runs/scale_two_instr8k_s10800_lr1e4/20261027-152516-b2b0c4b/final'
CONFIG = 'configs/contrastive_after_distill.yaml'
BASE = ['train.lr=1e-5', 'loss.anchor_cosine_weight=10', 'loss.anchor_gram_weight=200',
        f'data.pairs_train={DRIVE_ROOT}/data/stage1/pairs_train_clean_ans.jsonl']
FIXED = ['loss.answer_weight_mode=fixed']
RUNS = [
    ('ctr_ans_fixed01', BASE + FIXED + ['loss.answer_weight=0.1']),
    ('ctr_ans_fixed03', BASE + FIXED + ['loss.answer_weight=0.3']),
    ('ctr_ans_fixed05', BASE + FIXED + ['loss.answer_weight=0.5']),
    # dynamic: w = 0.1 + 0.4 * min(1, length / ref); TurHistQuAD spans ~0.14, WikiRAG sentences 0.5
    ('ctr_ans_words', BASE + ['loss.answer_weight_mode=words', 'loss.answer_ref_words=20']),
    ('ctr_ans_chars', BASE + ['loss.answer_weight_mode=chars', 'loss.answer_ref_chars=150']),
]
SHARES = []                               # share of the contrastive model in a soup with the distilled model; [] = no soups

import glob, os

assert os.path.isdir(DISTILLED), f'distilled model not found: {DISTILLED}'
finals = {}
for name, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    done = [d for d in run_dirs if os.path.isdir(d + 'final')]
    if done:
        print();print(f'########## {name}: already finished, skipped')
        finals[name] = done[-1] + 'final'
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}', f'student.hf_id={DISTILLED}'] + extra)
    print();print(f'########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train_contrastive --config {CONFIG} {run_id_arg} {overrides}')
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    finals[name] = run_dirs[-1] + 'final'
print(finals)

In [ ]:
# Soups: distilled model blended with each contrastive model; saved to Drive/models/ (evaluate with eval_trio presets).
from turkish_distill.soup import soup_models

for name, final in finals.items():
    for share in SHARES:
        out = f'{DRIVE_ROOT}/models/soup_{name}_c{int(round(share * 100))}'
        if os.path.exists(out + '/SOUP.json'):
            print('exists, skipped:', out)
            continue
        soup_models([DISTILLED, final], [1 - share, share], out, label=f'distilled + {name}')
        print('saved:', out)


In [ ]:
# Paste these into eval_trio's PRESETS (they are written out here so the paths need no guessing):
for name, final in finals.items():
    print(f"    '{name}': dict(model='{final}'),")
    for share in SHARES:
        print(f"    'soup_{name}_c{int(round(share * 100))}': dict(model=f'{{DRIVE_ROOT}}/models/soup_{name}_c{int(round(share * 100))}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()
